In [6]:
import pandas as pd

df = pd.read_excel("../data/Financial_Sample.xlsx")
print(df.shape)       # rows x columns
print(df.dtypes)      # column types
print(df.isnull().sum())  # missing values
df.head()

(700, 16)
Segment                           str
Country                           str
Product                           str
Discount Band                     str
Units Sold                    float64
Manufacturing Price             int64
Sale Price                      int64
Gross Sales                   float64
Discounts                     float64
 Sales                        float64
COGS                          float64
Profit                        float64
Date                   datetime64[us]
Month Number                    int64
Month Name                        str
Year                            int64
dtype: object
Segment                 0
Country                 0
Product                 0
Discount Band          53
Units Sold              0
Manufacturing Price     0
Sale Price              0
Gross Sales             0
Discounts               0
 Sales                  0
COGS                    0
Profit                  0
Date                    0
Month Number            0
Mont

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Canada,Carretera,NaN,1618.5,3,20,32370.0,0.0,32370.0,16185.0,16185.0,2014-01-01,1,January,2014
1,Government,Germany,Carretera,NaN,1321.0,3,20,26420.0,0.0,26420.0,13210.0,13210.0,2014-01-01,1,January,2014
2,Midmarket,France,Carretera,NaN,2178.0,3,15,32670.0,0.0,32670.0,21780.0,10890.0,2014-06-01,6,June,2014
3,Midmarket,Germany,Carretera,NaN,888.0,3,15,13320.0,0.0,13320.0,8880.0,4440.0,2014-06-01,6,June,2014
4,Midmarket,Mexico,Carretera,NaN,2470.0,3,15,37050.0,0.0,37050.0,24700.0,12350.0,2014-06-01,6,June,2014


1. Which segment drives the most revenue?
2. Which product has the highest profit margin?
3. Which country has the most sales growth?

In [7]:
# Q1: Which segment drives the most revenue?
segment_revenue = df.groupby('Segment')['Gross Sales'].sum().sort_values(ascending=False)
print("Revenue by Segment:")
print(segment_revenue)
print(f"\nTop segment: {segment_revenue.index[0]} with ${segment_revenue.values[0]:,.2f}")

Revenue by Segment:
Segment
Government          56403066.5
Small Business      45941700.0
Enterprise          21069000.0
Midmarket            2582670.0
Channel Partners     1935162.0
Name: Gross Sales, dtype: float64

Top segment: Government with $56,403,066.50


In [8]:
# Q2: Which product has the highest profit margin?
df['Profit Margin'] = (df['Profit'] / df['Gross Sales'] * 100).round(2)
product_margin = df.groupby('Product')['Profit Margin'].mean().sort_values(ascending=False)
print("Average Profit Margin by Product:")
print(product_margin)
print(f"\nHighest margin: {product_margin.index[0]} with {product_margin.values[0]:.2f}%")

Average Profit Margin by Product:
Product
Carretera    27.864086
Amarilla     26.761915
Paseo        26.538911
Montana      25.481613
Velo         24.641468
VTT          24.508716
Name: Profit Margin, dtype: float64

Highest margin: Carretera with 27.86%


In [9]:
# Q3: Which country has the most sales growth?
df['Year-Month'] = df['Date'].dt.to_period('M')
country_monthly_sales = df.groupby(['Country', 'Year-Month'])['Gross Sales'].sum().reset_index()
country_growth = []
for country in country_monthly_sales['Country'].unique():
    country_data = country_monthly_sales[country_monthly_sales['Country'] == country].sort_values('Year-Month')
    if len(country_data) > 1:
        first_month = country_data['Gross Sales'].iloc[0]
        last_month = country_data['Gross Sales'].iloc[-1]
        growth = ((last_month - first_month) / first_month * 100) if first_month > 0 else 0
        country_growth.append({'Country': country, 'Growth': growth})

growth_df = pd.DataFrame(country_growth).sort_values('Growth', ascending=False)
print("Sales Growth by Country (%):")
print(growth_df)
print(f"\nHighest growth: {growth_df.iloc[0]['Country']} with {growth_df.iloc[0]['Growth']:.2f}% growth")

Sales Growth by Country (%):
                    Country      Growth
0                    Canada  287.787591
1                    France  188.826703
3                    Mexico  160.638078
2                   Germany  103.630705
4  United States of America   87.285737

Highest growth: Canada with 287.79% growth


## Answers

### 1. Which segment drives the most revenue?
**Government** is the clear leader, generating **$56.4 million** in gross sales. This is significantly higher than the other segments:
- Government: $56.4M
- Small Business: $45.9M
- Enterprise: $21.1M
- Midmarket: $2.6M
- Channel Partners: $1.9M

### 2. Which product has the highest profit margin?
**Carretera** has the highest average profit margin at **27.86%**, followed closely by Amarilla (26.76%) and Paseo (26.54%). The profit margins are fairly consistent across products, ranging from 24.5% to 27.9%.

### 3. Which country has the most sales growth?
**Canada** shows exceptional sales growth of **287.79%** from the first to the last month in the dataset. This is followed by:
- France: 188.83%
- Mexico: 160.64%
- Germany: 103.63%
- USA: 87.29%